In [30]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

In [31]:
DATA_PATH = Path("train_features.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (7534, 16)


,rms_x,peak_x,kurtosis_x,skew_x,std_x,crest_x,rms_y,peak_y,kurtosis_y,skew_y,std_y,crest_y,time_step,total_steps,RUL,bearing
0,0.561746,2.010,-0.131465,-0.004711,0.561735,3.578132,0.435801,1.591,-0.035080,0.002713,0.435797,3.650745,0,2803,2802,Bearing1_1
1,0.535112,1.915,-0.084646,-0.025791,0.535083,3.578687,0.420968,1.666,0.150620,0.077380,0.420967,3.957542,1,2803,2801,Bearing1_1
2,0.531158,1.901,0.033388,-0.005016,0.531142,3.578971,0.425605,1.584,-0.061552,-0.024393,0.425600,3.721758,2,2803,2800,Bearing1_1
3,0.554833,1.910,0.043419,-0.080165,0.554830,3.442476,0.445524,1.600,-0.113319,-0.008190,0.445524,3.591275,3,2803,2799,Bearing1_1
4,0.566652,1.767,-0.185477,-0.034187,0.566646,3.118317,0.423847,1.373,-0.034816,0.081213,0.423731,3.239377,4,2803,2798,Bearing1_1


In [32]:
df["life_fraction"] = (
    df["time_step"] / df["total_steps"]
)

In [33]:
feature_cols = [
    "rms_x",
    "peak_x",
    "kurtosis_x",
    "skew_x",
    "std_x",
    "crest_x",
    "rms_y",
    "peak_y",
    "kurtosis_y",
    "skew_y",
    "std_y",
    "crest_y",
    "time_step",
    "life_fraction"
]

X = df[feature_cols]

y = df["RUL"]

groups = df["bearing"]

print("Feature shape:", X.shape)

Feature shape: (7534, 14)


In [34]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Train bearings:")
print(set(groups.iloc[train_idx]))

print("\nTest bearings:")
print(set(groups.iloc[test_idx]))

Train bearings:
{'Bearing2_2', 'Bearing3_2', 'Bearing3_1', 'Bearing2_1'}

Test bearings:
{'Bearing1_2', 'Bearing1_1'}


In [35]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

print("Training SVR Model...")

# Scale features (important for SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = SVR(
    kernel='rbf',
    C=100,
    gamma='scale',
    epsilon=0.1
)

model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\nSVR Performance:")
print(f"  RMSE: {rmse:.2f}")
print(f"  MAE: {mae:.2f}")
print(f"  R²: {r2:.4f}")

Training SVR Model...

SVR Performance:
  RMSE: 766.58
  MAE: 596.95
  R²: 0.1400


In [36]:
def phm_score(y_true, y_pred):

    score = 0

    for true, pred in zip(y_true, y_pred):

        d = pred - true

        if d < 0:
            score += np.exp(-d / 13) - 1

        else:
            score += np.exp(d / 10) - 1

    return score


score = phm_score(
    y_test.values,
    y_pred
)

print("PHM Score:", score)

PHM Score: 1.2232476895155174e+72


In [37]:
results = df.iloc[test_idx][[
    "bearing",
    "time_step",
    "total_steps"
]].copy()

results["Actual_RUL"] = y_test.values

results["Predicted_RUL"] = np.maximum(
    y_pred,
    0
)

residuals = y_test.values - y_pred

uncertainty = np.std(residuals)

results["Uncertainty"] = uncertainty

results["Lower_RUL"] = np.maximum(
    results["Predicted_RUL"] - 1.96 * uncertainty,
    0
)

results["Upper_RUL"] = (
    results["Predicted_RUL"] + 1.96 * uncertainty
)

results.head()

,bearing,time_step,total_steps,Actual_RUL,Predicted_RUL,Uncertainty,Lower_RUL,Upper_RUL
0,Bearing1_1,0,2803,2802,868.774783,508.193027,0.0,1864.833115
1,Bearing1_1,1,2803,2801,871.051478,508.193027,0.0,1867.109811
2,Bearing1_1,2,2803,2800,868.743247,508.193027,0.0,1864.801579
3,Bearing1_1,3,2803,2799,869.959287,508.193027,0.0,1866.017619
4,Bearing1_1,4,2803,2798,839.618893,508.193027,0.0,1835.677225


In [38]:
def classify_health(rul):

    if rul <= 100:
        return "Imminent failure"

    elif rul <= 500:
        return "Wear detectable"

    else:
        return "Non-critical"


results["Actual_Health_State"] = (
    results["Actual_RUL"]
    .apply(classify_health)
)

results["Predicted_Health_State"] = (
    results["Predicted_RUL"]
    .apply(classify_health)
)

results.head()

,bearing,time_step,total_steps,Actual_RUL,Predicted_RUL,Uncertainty,Lower_RUL,Upper_RUL,Actual_Health_State,Predicted_Health_State
0,Bearing1_1,0,2803,2802,868.774783,508.193027,0.0,1864.833115,Non-critical,Non-critical
1,Bearing1_1,1,2803,2801,871.051478,508.193027,0.0,1867.109811,Non-critical,Non-critical
2,Bearing1_1,2,2803,2800,868.743247,508.193027,0.0,1864.801579,Non-critical,Non-critical
3,Bearing1_1,3,2803,2799,869.959287,508.193027,0.0,1866.017619,Non-critical,Non-critical
4,Bearing1_1,4,2803,2798,839.618893,508.193027,0.0,1835.677225,Non-critical,Non-critical


In [39]:
results.to_csv(
    "svr_rul_predictions.csv",
    index=False
)

joblib.dump(
    model,
    "svr_rul_model.pkl"
)

joblib.dump(
    scaler,
    "svr_scaler.pkl"
)

print("Saved svr_rul_predictions.csv")
print("Saved svr_rul_model.pkl")
print("Saved svr_scaler.pkl")

Saved svr_rul_predictions.csv
Saved svr_rul_model.pkl
Saved svr_scaler.pkl


In [40]:
print("="*70)
print("SVR MODEL SUMMARY")
print("="*70)

print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R2 Score: {r2:.3f}")
print(f"PHM Score: {score:.2f}")

print("Saved model: svr_rul_model.pkl")

print("="*70)

SVR MODEL SUMMARY
RMSE: 766.58
MAE: 596.95
R2 Score: 0.140
PHM Score: 1223247689515517379118343170384135245692538299038450956669034950708690944.00
Saved model: svr_rul_model.pkl


Train Final SVR Model on All Training Bearings

After validation, the final SVR model is retrained using all available training bearings.

This final model is then used for prediction on the real unseen bearings inside the `Test_set` folder.

In [41]:
# Scale ALL training data
final_scaler = StandardScaler()

X_scaled_all = final_scaler.fit_transform(X)

# Train final SVR model on ALL bearings
final_model = SVR(
    kernel="rbf",
    C=100,
    gamma="scale",
    epsilon=0.1
)

print("Training final SVR model on all training bearings...")

final_model.fit(X_scaled_all, y)

print("Final SVR model training complete!")

# Replace old validation model with final model
model = final_model

# Replace old scaler with final scaler
scaler = final_scaler

Training final SVR model on all training bearings...
Final SVR model training complete!


Predict on Real Test Set Bearings

The SVR model is now applied to the real unseen bearings inside the `Test_set` folder.

This step fixes the issue of only validating on training-set bearings. The final output gives RUL prediction and health-state classification for each real test bearing.

In [42]:
import os
from scipy.stats import kurtosis, skew

def extract_features(filepath):

    raw_df = pd.read_csv(filepath, header=None)

    if raw_df.shape[1] >= 6:
        ax = raw_df.iloc[:, 4].values
        ay = raw_df.iloc[:, 5].values

    elif raw_df.shape[1] >= 2:
        ax = raw_df.iloc[:, -2].values
        ay = raw_df.iloc[:, -1].values

    else:
        raise ValueError(f"Too few columns: {filepath}")

    features = {}

    for name, sig in [("x", ax), ("y", ay)]:

        features[f"rms_{name}"] = np.sqrt(np.mean(sig**2))
        features[f"peak_{name}"] = np.max(np.abs(sig))
        features[f"kurtosis_{name}"] = kurtosis(sig)
        features[f"skew_{name}"] = skew(sig)
        features[f"std_{name}"] = np.std(sig)
        features[f"crest_{name}"] = (
            np.max(np.abs(sig)) /
            (np.sqrt(np.mean(sig**2)) + 1e-10)
        )

    return features


test_path = "Test_set"

test_bearings = sorted([
    folder for folder in os.listdir(test_path)
    if folder.startswith("Bearing")
])

print("Test bearings found:", test_bearings)

all_test = []

for bearing in test_bearings:

    bearing_path = os.path.join(test_path, bearing)

    files = sorted([
        file for file in os.listdir(bearing_path)
        if file.endswith(".csv") and not file.lower().startswith("temp")
    ])

    print(f"Processing {bearing}: {len(files)} vibration files")

    records = []

    for i, file in enumerate(files):

        file_path = os.path.join(bearing_path, file)

        try:
            feats = extract_features(file_path)

        except Exception as e:
            print(f"Skipping {file_path}: {e}")
            continue

        feats["time_step"] = i
        feats["total_steps"] = len(files)
        feats["bearing"] = bearing

        records.append(feats)

        if i % 100 == 0:
            print(f"  {bearing}: processed {i}/{len(files)}")

    bearing_df = pd.DataFrame(records)

    all_test.append(bearing_df)

    print(f"Finished {bearing}")

test_df = pd.concat(all_test, ignore_index=True)

print("Test set shape:", test_df.shape)

test_df.head()

Test bearings found: ['Bearing1_3', 'Bearing1_4', 'Bearing1_5', 'Bearing1_6', 'Bearing1_7', 'Bearing2_3', 'Bearing2_4', 'Bearing2_5', 'Bearing2_6', 'Bearing2_7', 'Bearing3_3']
Processing Bearing1_3: 1802 vibration files
  Bearing1_3: processed 0/1802
  Bearing1_3: processed 100/1802
  Bearing1_3: processed 200/1802
  Bearing1_3: processed 300/1802
  Bearing1_3: processed 400/1802
  Bearing1_3: processed 500/1802
  Bearing1_3: processed 600/1802
  Bearing1_3: processed 700/1802
  Bearing1_3: processed 800/1802
  Bearing1_3: processed 900/1802
  Bearing1_3: processed 1000/1802
  Bearing1_3: processed 1100/1802
  Bearing1_3: processed 1200/1802
  Bearing1_3: processed 1300/1802
  Bearing1_3: processed 1400/1802
  Bearing1_3: processed 1500/1802
  Bearing1_3: processed 1600/1802
  Bearing1_3: processed 1700/1802
  Bearing1_3: processed 1800/1802
Finished Bearing1_3
Processing Bearing1_4: 1139 vibration files
  Bearing1_4: processed 0/1139
  Bearing1_4: processed 100/1139
  Bearing1_4: proc

,rms_x,peak_x,kurtosis_x,skew_x,std_x,crest_x,rms_y,peak_y,kurtosis_y,skew_y,std_y,crest_y,time_step,total_steps,bearing
0,0.415616,1.478,0.068604,-0.004453,0.415615,3.556165,0.302195,1.081,0.045011,-0.009074,0.301856,3.577161,0,1802,Bearing1_3
1,0.391144,1.513,0.196395,0.027562,0.391144,3.868142,0.305312,1.108,-0.012812,-0.014064,0.305267,3.629077,1,1802,Bearing1_3
2,0.389165,1.602,0.380518,0.073442,0.388961,4.116510,0.301042,1.207,0.109585,0.066631,0.300618,4.009405,2,1802,Bearing1_3
3,0.380670,1.624,0.489310,-0.092060,0.380567,4.266161,0.296139,1.288,0.297545,0.008303,0.295927,4.349303,3,1802,Bearing1_3
4,0.400809,1.595,0.268491,-0.054019,0.400785,3.979449,0.300955,1.146,0.002069,0.075465,0.300520,3.807880,4,1802,Bearing1_3


In [43]:
test_df["life_fraction"] = (
    test_df["time_step"] / test_df["total_steps"]
)

test_feature_cols = feature_cols

X_real_test = test_df[test_feature_cols]

# IMPORTANT: SVR needs scaled features
X_real_test_scaled = scaler.transform(X_real_test)

y_pred_test = model.predict(X_real_test_scaled)

test_df["Predicted_RUL"] = np.maximum(y_pred_test, 0)

test_df["Uncertainty"] = uncertainty

test_df["Lower_RUL"] = np.maximum(
    test_df["Predicted_RUL"] - 1.96 * uncertainty,
    0
)

test_df["Upper_RUL"] = (
    test_df["Predicted_RUL"] + 1.96 * uncertainty
)

test_df["Health_State"] = (
    test_df["Predicted_RUL"]
    .apply(classify_health)
)

test_df.to_csv(
    "svr_test_predictions_timeseries.csv",
    index=False
)

print("Saved svr_test_predictions_timeseries.csv")

test_df.head()

Saved svr_test_predictions_timeseries.csv


,rms_x,peak_x,kurtosis_x,skew_x,std_x,crest_x,rms_y,peak_y,kurtosis_y,skew_y,...,crest_y,time_step,total_steps,bearing,life_fraction,Predicted_RUL,Uncertainty,Lower_RUL,Upper_RUL,Health_State
0,0.415616,1.478,0.068604,-0.004453,0.415615,3.556165,0.302195,1.081,0.045011,-0.009074,...,3.577161,0,1802,Bearing1_3,0.000000,1558.207047,508.193027,562.148715,2554.265380,Non-critical
1,0.391144,1.513,0.196395,0.027562,0.391144,3.868142,0.305312,1.108,-0.012812,-0.014064,...,3.629077,1,1802,Bearing1_3,0.000555,1554.224514,508.193027,558.166182,2550.282846,Non-critical
2,0.389165,1.602,0.380518,0.073442,0.388961,4.116510,0.301042,1.207,0.109585,0.066631,...,4.009405,2,1802,Bearing1_3,0.001110,1526.656216,508.193027,530.597883,2522.714548,Non-critical
3,0.380670,1.624,0.489310,-0.092060,0.380567,4.266161,0.296139,1.288,0.297545,0.008303,...,4.349303,3,1802,Bearing1_3,0.001665,1507.985062,508.193027,511.926730,2504.043395,Non-critical
4,0.400809,1.595,0.268491,-0.054019,0.400785,3.979449,0.300955,1.146,0.002069,0.075465,...,3.807880,4,1802,Bearing1_3,0.002220,1539.147257,508.193027,543.088925,2535.205589,Non-critical


In [44]:
latest = (
    test_df
    .sort_values("time_step")
    .groupby("bearing")
    .last()
    .reset_index()
)

latest = latest[[
    "bearing",
    "time_step",
    "total_steps",
    "Predicted_RUL",
    "Uncertainty",
    "Lower_RUL",
    "Upper_RUL",
    "Health_State"
]]

latest.to_csv(
    "svr_test_predictions.csv",
    index=False
)

print("Saved svr_test_predictions.csv")

print("\n=== CURRENT BEARING STATUS USING SVR ===")

for _, row in latest.iterrows():

    print(f"\nBearing: {row['bearing']}")
    print(f"Health State : {row['Health_State']}")
    print(f"Predicted RUL: {row['Predicted_RUL']:.0f} steps")
    print(f"Uncertainty  : ±{row['Uncertainty']:.0f} steps")

    if row["Health_State"] == "Imminent failure":
        print("Action: Replace immediately.")

    elif row["Health_State"] == "Wear detectable":
        print("Action: Schedule maintenance soon.")

    else:
        print("Action: No immediate action needed.")

latest

Saved svr_test_predictions.csv

=== CURRENT BEARING STATUS USING SVR ===

Bearing: Bearing1_3
Health State : Wear detectable
Predicted RUL: 343 steps
Uncertainty  : ±508 steps
Action: Schedule maintenance soon.

Bearing: Bearing1_4
Health State : Wear detectable
Predicted RUL: 245 steps
Uncertainty  : ±508 steps
Action: Schedule maintenance soon.

Bearing: Bearing1_5
Health State : Imminent failure
Predicted RUL: 0 steps
Uncertainty  : ±508 steps
Action: Replace immediately.

Bearing: Bearing1_6
Health State : Imminent failure
Predicted RUL: 0 steps
Uncertainty  : ±508 steps
Action: Replace immediately.

Bearing: Bearing1_7
Health State : Imminent failure
Predicted RUL: 28 steps
Uncertainty  : ±508 steps
Action: Replace immediately.

Bearing: Bearing2_3
Health State : Wear detectable
Predicted RUL: 101 steps
Uncertainty  : ±508 steps
Action: Schedule maintenance soon.

Bearing: Bearing2_4
Health State : Imminent failure
Predicted RUL: 62 steps
Uncertainty  : ±508 steps
Action: Replace 

,bearing,time_step,total_steps,Predicted_RUL,Uncertainty,Lower_RUL,Upper_RUL,Health_State
0,Bearing1_3,1801,1802,343.018870,508.193027,0.0,1339.077203,Wear detectable
1,Bearing1_4,1138,1139,245.089337,508.193027,0.0,1241.147670,Wear detectable
2,Bearing1_5,2301,2302,0.000000,508.193027,0.0,996.058332,Imminent failure
3,Bearing1_6,2301,2302,0.000000,508.193027,0.0,996.058332,Imminent failure
4,Bearing1_7,1501,1502,28.453713,508.193027,0.0,1024.512046,Imminent failure
5,Bearing2_3,1201,1202,101.017735,508.193027,0.0,1097.076067,Wear detectable
6,Bearing2_4,611,612,62.408415,508.193027,0.0,1058.466748,Imminent failure
7,Bearing2_5,2001,2002,0.000000,508.193027,0.0,996.058332,Imminent failure
8,Bearing2_6,571,572,120.496453,508.193027,0.0,1116.554785,Wear detectable
9,Bearing2_7,171,172,0.000000,508.193027,0.0,996.058332,Imminent failure
